# 基于深度学习与统计方法的数据挖掘综合实验分析


## 滕啸啸
### 计算机学院25060115

---

### Abstract
**Abstract:** Data mining involves extracting valuable patterns from large-scale datasets. This paper presents a comprehensive study covering four distinct data mining tasks: image clustering, image anomaly detection, time series prediction, and unsupervised disease diagnosis. First, for the image clustering task, we utilized ResNet-50 for feature extraction and K-Means for clustering, achieving an ARI of 1.0 on the Cluster dataset. Second, we designed a Convolutional Autoencoder (CAE) for image anomaly detection, effectively identifying defects in industrial products. Third, we implemented a Multilayer Perceptron (MLP) to predict outdoor temperature changes based on historical meteorological data, achieving an R2 score of 0.9637. Finally, we proposed an unsupervised Autoencoder-based method for thyroid disease detection, which attained an AUC of 0.9448 using only normal samples for training. Experimental results demonstrate the effectiveness of the proposed methods across different domains.

**Keywords:** Data Mining; Clustering; Anomaly Detection; Time Series Prediction; Deep Learning

### 摘要
**摘要：** 数据挖掘旨在从大规模数据集中提取有价值的模式。本文报告了涵盖四个不同领域的综合数据挖掘实验：图像聚类、图像异常检测、时间序列预测以及无监督疾病诊断。首先，针对图像聚类任务，我们利用 ResNet-50 提取特征并结合 K-Means 算法，在 Cluster 数据集上实现了 ARI 为 1.0 的完美聚类。其次，我们设计了卷积自编码器（CAE）用于图像异常检测，有效识别了工业产品中的缺陷。第三，针对气象数据，我们构建了多层感知机（MLP）模型，基于历史数据准确预测了室外温度变化，拟合优度（R2）达到 0.9637。最后，我们提出了一种基于自编码器的无监督甲状腺疾病检测方法，仅使用正常样本训练即达到了 0.9448 的 AUC。实验结果验证了上述方法在处理多模态数据挖掘任务中的有效性与鲁棒性。

**关键词：** 数据挖掘；聚类；异常检测；时间序列预测；深度学习

---

# 1. 聚类任务 (Clustering Task)

在本节中，我们针对给定的图像数据集进行无监督聚类分析。数据集包含 cable, tile, bottle, pill, leather, transistor 共 6 个类别，每类 100 张图片，总计 600 张图片。我们的目标是在不使用标签信息的情况下，将图像正确地划分到各自的语义类别中。

## 1.0 问题的形式化描述
给定一个包含 $N$ 张图像的数据集 $X = \{x_1, x_2, ..., x_N\}$，其中 $N=600$。假设数据集中存在 $K$ 个潜在的语义类别（在本任务中 $K=6$）。

聚类问题的目标是寻找一个划分 $C = \{C_1, C_2, ..., C_K\}$，使得每个数据点 $x_i$ 属于且仅属于一个簇 $C_j$，同时最小化簇内方差（Intra-cluster Variance）并最大化簇间方差（Inter-cluster Variance）。

如果我们采用基于距离的划分方法（如 K-Means），其核心优化目标可以形式化定义为最小化平方误差和（Sum of Squared Errors, SSE）：

$$
J = \sum_{j=1}^{K} \sum_{x_i \in C_j} ||f(x_i) - \mu_j||^2
$$

其中：
* $f(x_i)$ 是图像 $x_i$ 经过特征提取后的高维特征向量；
* $\mu_j$ 是第 $j$ 个簇的中心（Centroid），即 $\mu_j = \frac{1}{|C_j|} \sum_{x_i \in C_j} f(x_i)$。

## 1.1 图像特征处理
由于原始图像数据处于高维像素空间（Pixel Space），直接在像素层面进行欧氏距离计算往往受到“维度灾难”和背景噪声的影响，无法有效捕捉语义信息。因此，我们设计了由 **深度特征提取** 和 **统计降维** 构成的两阶段特征处理流。

### 1.1.1 深度特征提取 (Deep Feature Extraction)
我们利用 **迁移学习 (Transfer Learning)** 的思想，使用在 ImageNet 数据集上预训练的 **ResNet-50** 模型作为特征提取器。
具体步骤如下：
1.  **预处理**：将图像调整为 $224 \times 224$ 分辨率，并转化为 Tensor。随后进行标准化处理，减去 ImageNet 的均值 $[0.485, 0.456, 0.406]$ 并除以标准差 $[0.229, 0.224, 0.225]$，以符合模型输入分布。
2.  **前向传播**：移除 ResNet-50 模型的全连接分类层（FC Layer），保留卷积层和池化层作为骨干网络（Backbone）。
3.  **特征获取**：将图像输入网络，提取全局平均池化层（Global Average Pooling）的输出。

通过此过程，我们将每张图像 $x_i$ 映射为一个 2048 维的特征向量 $v_i \in \mathbb{R}^{2048}$。该向量富含图像的高层语义信息。

### 1.1.2 特征降维 (Dimensionality Reduction)
考虑到 2048 维的特征空间对于只有 600 个样本的数据集来说过于稀疏，且可能包含冗余信息，我们采用 **主成分分析 (PCA)** 进行降维。
* **输入**：$600 \times 2048$ 的特征矩阵。
* **策略**：保留 $95\%$ 的解释方差比（Explained Variance Ratio）。
* **结果**：特征维度从 2048 维显著降低至 **19 维**。这不仅降低了计算复杂度，还去除了特征中的噪声。

## 1.2 聚类算法选择
基于降维后的特征空间，我们选择 **K-Means 聚类算法**。
选择理由如下：
1.  **假设匹配**：PCA 降维后的特征通常服从混合高斯分布，符合 K-Means 对簇形状为凸形（Convex）和各向同性（Isotropic）的假设。
2.  **效率**：K-Means 算法的时间复杂度为 $O(N \cdot K \cdot D \cdot I)$，在大样本和低维度（19维）下收敛速度极快。

**算法配置：**
* **簇数量 ($K$)**：设定为 6，对应数据集的 6 个真实类别。
* **初始化策略**：使用 **k-means++** 算法初始化簇中心，以加速收敛并避免陷入局部最优。
* **最大迭代次数**：默认设置（通常为 300），直至中心点不再发生显著位移。

我们利用 `sklearn.cluster.KMeans` 在 M 芯片 Mac 环境下完成了模型训练与预测。

## 1.3 聚类效果评估
为了定量评估聚类性能，我们利用数据集提供的真实标签（Ground Truth）计算了以下两个指标：

1.  **调整兰德系数 (Adjusted Rand Index, ARI)**：
    衡量聚类结果与真实标签的一致性，取值范围 $[-1, 1]$。公式如下：
    $$
    ARI = \frac{RI - E[RI]}{\max(RI) - E[RI]}
    $$
    其中 $RI$ 为兰德系数。
    
2.  **归一化互信息 (Normalized Mutual Information, NMI)**：
    基于信息论的度量，衡量两个分布的共享信息量。
    $$
    NMI(Y, C) = \frac{2 \cdot I(Y; C)}{H(Y) + H(C)}
    $$

**实验结果：**

| 评估指标 | 实验得分 | 说明 |
| :--- | :---: | :--- |
| **ARI** | **1.0000** | 聚类结果与真实标签完全一致 |
| **NMI** | **1.0000** | 预测分布完美复原了真实分布 |

**可视化验证：**
我们进一步使用 **t-SNE** 算法将 19 维特征降至 2 维进行可视化（如图所示）。从图中可以清晰地看到，6 个类别的样本被完美地分成了 6 个独立的簇，簇内紧凑且簇间分离度极高，验证了我们特征提取策略的有效性。

![t-SNE Visualization](cluster_tsne.png)
*图：基于 ResNet50 特征与 PCA 降维后的 t-SNE 聚类可视化*

# 2. 图像异常检测任务 (Image Anomaly Detection Task)

在本节中，我们针对工业检测场景（如 Hazelnut 和 Zipper）设计并实现了一种基于重构误差的图像异常检测算法。由于在实际应用中，异常样本通常难以获取且种类繁多，而正常样本相对充足，因此我们采用**无监督学习**的思路，仅利用正常样本进行模型训练。

## 2.1 问题的形式化描述
图像异常检测的目标是学习一个能够区分正常图像与异常图像的判别函数。
给定一个仅包含正常样本的训练数据集 $\mathcal{D}_{train} = \{x_1, x_2, ..., x_N\}$，其中 $x_i \in \mathbb{R}^{H \times W \times C}$ 为正常图像。

我们需要构建一个异常评分函数 $A(x)$。对于任意新的测试样本 $x_{test}$：
* 如果 $x_{test}$ 是正常的，$A(x_{test})$ 应较小（低于阈值 $\tau$）；
* 如果 $x_{test}$ 包含异常（如缺陷、裂纹），$A(x_{test})$ 应较大（高于阈值 $\tau$）。

形式化地，我们的目标是：
$$
\text{Prediction}(x) = 
\begin{cases} 
\text{Normal}, & \text{if } A(x) < \tau \\
\text{Anomaly}, & \text{if } A(x) \geq \tau 
\end{cases}
$$

其中，我们在本任务中采用 **重构误差 (Reconstruction Error)** 作为异常评分函数 $A(x)$。

## 2.2 图像特征处理
为了适配深度神经网络的输入要求，我们对原始图像数据进行了以下预处理操作：

1.  **尺寸归一化 (Resizing)**：
    原始图像尺寸不一，我们将所有输入图像统一调整为 $224 \times 224$ 像素分辨率。这既保留了足够的纹理细节以便检测细微缺陷，又控制了计算量。
    
2.  **张量转换 (Tensor Conversion)**：
    利用 PyTorch 的 `transforms.ToTensor()` 将图像数据从 $[0, 255]$ 的整数空间映射到 $[0, 1]$ 的浮点数空间，并将通道顺序调整为 $(C, H, W)$，即 $(3, 224, 224)$。

3.  **数据加载 (Data Loading)**：
    对于训练阶段，我们仅筛选标记为 "good" 的正常样本；对于测试阶段，我们混合加载正常样本与包含各类缺陷（如 "hole", "crack" 等）的异常样本，以验证模型的鲁棒性。

## 2.3 异常检测模型设计
我们设计了一个 **卷积自编码器 (Convolutional Autoencoder, CAE)** 模型。该模型的设计核心思想是：**自编码器通过学习将输入压缩为低维编码再重构回原图，如果仅使用正常样本训练，模型将“学会”如何重构正常纹理；当输入异常图像时，模型因未见过此类模式而无法有效重构，从而产生高重构误差。**

### 2.3.1 模型架构
模型由 **编码器 (Encoder)** 和 **解码器 (Decoder)** 两部分组成：

1.  **编码器 $E(x)$**：
    负责提取图像的潜在特征（Latent Features）。它包含 4 个卷积块，每个块由 `Conv2d`、`BatchNorm2d` 和 `LeakyReLU` 激活函数组成。
    * 输入维度：$3 \times 224 \times 224$
    * 通过逐步下采样（Stride=2），最终将图像压缩为 $256 \times 14 \times 14$ 的高维特征向量 $z$。

2.  **解码器 $D(z)$**：
    负责将潜在特征重构回原始图像空间。它由 4 个转置卷积块（`ConvTranspose2d`）组成，逐步上采样恢复图像尺寸。
    * 最后一层使用 `Sigmoid` 激活函数，确保输出值限制在 $[0, 1]$ 范围内，与输入图像分布一致。

### 2.3.2 损失函数与异常评分
我们使用 **均方误差 (Mean Squared Error, MSE)** 作为训练时的损失函数，旨在最小化输入 $x$ 与重构输出 $\hat{x}$ 之间的差异：

$$
\mathcal{L} = \frac{1}{N} \sum_{i=1}^{N} ||x_i - D(E(x_i))||^2
$$

在**检测阶段**，对于测试图像 $x$，其异常分数定义为该图像所有像素点的重构误差之和：

$$
A(x) = \sum_{c, h, w} (x^{(c,h,w)} - \hat{x}^{(c,h,w)})^2
$$

## 2.4 实验评估与结果
我们在 **Image_Anomaly_Detection** 数据集的两个子集（Hazelnut 和 Zipper）上进行了实验。模型使用 Adam 优化器（学习率 $1e-3$）训练了 100 个 Epoch。

我们采用 **AUROC (Area Under the Receiver Operating Characteristic curve)** 作为主要评估指标。AUROC 能够衡量模型在不同阈值下区分正常与异常样本的能力，值越接近 1 表示性能越好。

### 2.4.1 实验结果

| 类别 (Category) | 训练样本数 | 测试样本数 (异常数) | AUROC 得分 | 结果分析 |
| :--- | :---: | :---: | :---: | :--- |
| **Hazelnut** (榛子) | 200 | 55 (15) | **0.8750** | 模型表现优异，能够有效通过纹理差异识别表面缺陷。 |
| **Zipper** (拉链) | 200 | 47 (15) | **0.6125** | 模型表现一般。可能是因为拉链的结构比榛子更复杂（包含金属与织物），且异常（如拉链齿损坏）较为细微，简单的 MSE 损失难以捕捉。 |

### 2.4.2 损失收敛情况
训练过程中的 Loss 曲线显示模型在 100 个 Epoch 内成功收敛（Loss 降至 $10^{-4}$ 数量级），表明自编码器成功学习到了正常样本的数据分布。

### 2.4.3 异常分数分布可视化
通过绘制测试集中正常样本与异常样本的重构误差分布直方图（如图所示），我们观察到：
* 在 **Hazelnut** 数据集中，正常样本（绿色）的重构误差明显集中在较低区间，而异常样本（红色）的误差分布较高，两者分离度较好。
* 在 **Zipper** 数据集中，两类样本的误差分布存在一定重叠，导致 AUROC 分数相对较低。

![Anomaly Score Distribution - Hazelnut](hazelnut.png)
*图：Hazelnut 类别的异常分数分布 (AUROC=0.875)*
![Anomaly Score Distribution - Zipper](zipper.png)
*图：Hazelnut 类别的异常分数分布 (AUROC=0.6125)*

# 3. 时间序列预测任务 (Time Series Prediction Task)

在本节中，我们利用德国耶拿气象站（Jena Climate）的数据集，设计并实现了一个基于神经网络的时间序列预测模型。任务目标是根据过去 2 小时（即 12 个时间步，每 10 分钟一个采样点）的气象数据，预测下一时刻的室外温度（OT）。

## 3.0 问题的形式化描述
给定一个多变量时间序列数据集 $\mathcal{D} = \{X_1, X_2, ..., X_N\}$，其中 $X_t \in \mathbb{R}^{D}$ 表示第 $t$ 个时刻的气象特征向量，$D=21$ 为特征维度。

我们的目标是训练一个非线性函数逼近器 $f_{\theta}$（即 MLP 神经网络），利用过去 $w$ 个时间步的历史观测值来预测未来第 $t+1$ 个时刻的目标变量 $y_{t+1}$（即 OT 值）：

$$
\hat{y}_{t+1} = f_{\theta}(X_{t-w+1}, X_{t-w+2}, ..., X_t)
$$

在本任务中，滑动窗口大小设定为 $w=12$（对应 2 小时）。我们将输入的时间序列窗口展平（Flatten）为一个高维特征向量，输入维度为 $12 \times 21 = 252$。因此，问题转化为一个典型的**监督回归（Supervised Regression）**问题。

## 3.1 数据预处理与划分
神经网络对输入数据的分布非常敏感，为了加速收敛并防止梯度问题，我们进行了以下预处理：

1.  **缺失值处理**：
    采用 **线性插值 (Linear Interpolation)** 方法对原始数据中的缺失值进行填充，保证时间序列的连续性。

2.  **特征标准化 (Standardization)**：
    使用 **Z-Score 标准化**（StandardScaler）对所有特征进行变换，使其均值为 0，方差为 1。
    $$
    x' = \frac{x - \mu}{\sigma}
    $$
    这一步对于 MLP 至关重要，因为它能确保不同量纲的特征（如气压和风速）在网络权重更新时具有相似的梯度尺度。

3.  **滑动窗口切分 (Sliding Windowing)**：
    * **窗口大小**：12（过去 120 分钟）
    * **预测步长**：1（未来 10 分钟）
    * 最终生成的特征维度为 $(N_{samples}, 252)$。

4.  **数据集划分**：
    按照时间顺序（不进行 Shuffle），将前 **80%** 的数据（约 20,948 条样本）作为**训练集**，后 **20%** 的数据（约 5,228 条样本）作为**测试集**。

## 3.2 预测模型设计
为了捕捉气象数据中复杂的非线性关系和特征间的交互作用，我们选择 **多层感知机 (MLP)** 作为预测模型。

### 3.2.1 模型架构
我们构建了一个包含隐含层的全连接神经网络结构：

1.  **输入层 (Input Layer)**：
    * 节点数：252（对应 12 个时间步 $\times$ 21 个特征）。
    * 作用：接收展平后的历史气象窗口向量。

2.  **隐含层 (Hidden Layers)**：
    * 结构：包含若干个神经元的全连接层（Dense Layer）。
    * 激活函数：采用 **ReLU (Rectified Linear Unit)**。公式为 $f(x) = \max(0, x)$。
    * 作用：通过非线性变换提取数据中的高阶特征，捕捉气象变量（如“湿度增加导致温度变化滞后”）之间的复杂耦合关系。

3.  **输出层 (Output Layer)**：
    * 节点数：1。
    * 激活函数：无（Linear），直接输出回归预测值 $\hat{y}$。

### 3.2.2 设计思路
* **非线性建模能力**：与线性回归不同，MLP 具备万能逼近能力（Universal Approximation Theorem），能够拟合气象数据中潜在的非线性动态变化。
* **端到端学习**：通过反向传播算法（Backpropagation）和优化器（如 Adam），网络能够自动学习到哪些历史时刻或哪些特征组合对预测未来温度最为关键。

## 3.3 实验评估与结果
模型在测试集上的表现如下：

| 评估指标 | 测试集得分 | 结果分析 |
| :--- | :---: | :--- |
| **RMSE** | **4.3092** | 均方根误差约为 4.3，在温度预测任务中属于较好的精度范围。 |
| **MAE** | **2.9174** | 平均绝对误差小于 3 度，说明模型对大多数样本的预测偏差较小。 |
| **$R^2$ Score** | **0.9637** | 极高的拟合优度，表明 MLP 模型成功捕获了数据中 96% 以上的方差变化，证明了非线性模型在处理该时间序列任务上的有效性。 |

### 3.3.2 预测可视化
下图展示了 MLP 模型在测试集前 200 个时间步的预测结果。可以看出，预测曲线（红色虚线）紧密跟随真实曲线（蓝色实线），不仅准确预测了温度的整体趋势，对局部的波动细节也有较好的拟合能力。

![Outdoor Temperature Prediction](weather.png)
*图：室外温度 (OT) 预测结果对比 (前 200 个测试点)*

# 4. 无监督疾病判断任务 (Unsupervised Disease Diagnosis)

在本节中，我们针对甲状腺疾病数据集 (Thyroid)，设计并实现了一种基于 **深度自动编码器 (Autoencoder)** 的无监督异常检测模型。针对训练集仅包含正常样本（Label=0）的数据特点，我们将疾病诊断视为一个 **单类分类 (One-Class Classification)** 问题。

## 4.0 问题的形式化描述

* **数据集定义**：给定数据集 $X \in \mathbb{R}^{N \times 6}$，每个样本包含 6 个经过脱敏处理的生理特征。
* **训练集 ($\mathcal{D}_{train}$)**：仅包含 **正常样本**（Label=0），即 $\mathcal{D}_{train} = \{x_i | y_i = 0\}$。
* **测试集 ($\mathcal{D}_{test}$)**：包含 **正常样本** 与 **患病样本**（Label=1）。
* **目标**：学习一个映射函数 $f(x)$ 和异常评分函数 $S(x)$。对于测试样本 $x_{test}$，设定阈值 $\tau$ 进行判别：
    $$
    \hat{y} = \begin{cases} 
    1 \text{ (患病/异常)}, & \text{if } S(x_{test}) > \tau \\
    0 \text{ (正常)}, & \text{if } S(x_{test}) \leq \tau 
    \end{cases}
    $$
    在本任务中，我们使用 **重构误差 (Reconstruction Error)** 作为异常评分 $S(x)$。

## 4.1 无监督方法选择与理由

我们选择 **深度自动编码器 (Deep Autoencoder)** 作为核心算法。

**选择理由：**
1.  **数据分布适用性**：训练集中不存在患病样本（负样本），导致传统的监督学习算法（如 SVM、随机森林）无法通过学习决策边界来区分正负类。自动编码器属于无监督学习，仅需正常数据即可训练，完美契合“仅有正常样本”的场景。
2.  **流形学习假设**：自动编码器通过将高维输入压缩到低维潜空间（Latent Space）再还原，强制模型捕捉正常数据的核心相关性和流形分布。
3.  **判别机制有效性**：
    * 模型在正常样本上训练，能以极低的误差重构它们。
    * **患病样本** 的特征组合偏离了正常生理规律（即不在正常数据的流形上），模型无法有效还原，从而产生显著的 **高重构误差**。该误差天然适合作为区分正常与异常的指标。

## 4.2 模型实现与训练

### 4.2.1 模型架构
基于 PyTorch 构建了一个轻量级的全连接自动编码器（MLP-AE）：
* **编码器 (Encoder)**：`Input(6) -> Linear(12) -> Tanh -> Linear(3)`。将 6 维特征压缩为 3 维潜在向量，提取核心生理模式。
* **解码器 (Decoder)**：`Linear(3) -> Tanh -> Linear(12) -> Tanh -> Linear(6)`。将潜在向量还原回原始维度。

### 4.2.2 训练细节
* **环境**：在 Apple M 系列芯片上启用 **MPS (Metal Performance Shaders)** 加速训练。
* **损失函数**：均方误差 (MSE)，$\mathcal{L} = \frac{1}{N} \sum ||x - \hat{x}||^2$。
* **优化器**：Adam 优化器 (LR=0.01)。
* **训练过程**：仅使用训练集（正常样本）迭代 50 个 Epoch。
    * *Loss 变化*：从初始的 **0.4439** 下降至 **0.3385**，表明模型成功收敛并学习到了正常样本的特征分布。

## 4.3 评估模型效果

模型在测试集（共 1933 个样本，含 94 个患病样本）上进行了评估。我们计算每个测试样本的重构误差，并以此为依据进行分类。

### 4.3.1 核心指标
我们主要关注 **AUC (Area Under Curve)**，因为它能客观反映模型在不同阈值下的综合分类能力，不受样本不平衡影响。

* **AUC Score**: **0.9448**
    * *分析*：AUC 接近 0.95，远高于随机猜测的 0.5，说明重构误差能极好地区分患病与不患病样本。
* **最佳阈值 ($\tau$)**: **2.5396**
* **最佳 F1-Score**: **0.6893**

### 4.3.2 结果可视化分析
下图展示了模型在测试集上的详细表现：

![重构误差分布与ROC曲线](thyroid.png)

1.  **重构误差分布图 (左图)**：
    * **蓝色区域 (Normal)**：正常样本的重构误差非常集中，绝大多数分布在 [0, 1.5] 的低误差区间。
    * **红色区域 (Disease)**：患病样本的重构误差呈现明显的长尾分布，主要集中在 [2.5, 10+] 的高误差区间。
    * **绿色虚线 (Threshold)**：最佳阈值 ($\tau \approx 2.54$) 处于两个分布的波谷位置，清晰地将大部分正常样本与患病样本分开。

2.  **ROC 曲线 (右图)**：
    * 曲线呈现出完美的“拱形”，紧贴左上角。
    * 这意味着模型在保证极高 **检出率 (True Positive Rate)** 的同时，能够维持极低的 **误报率 (False Positive Rate)**。

### 结论
实验结果表明，基于自动编码器的无监督异常检测方法在甲状腺疾病诊断任务中表现优异。模型仅通过学习“正常生理指标”的规律，即可有效识别出偏离该规律的“患病样本”，AUC 达到 0.9448，验证了该方法在医疗辅助诊断中的有效性。

## 5. 总结 (Conclusion)
本文综合运用了深度学习（ResNet, CAE, MLP, Autoencoder）与统计方法（K-Means, PCA），成功解决了聚类、异常检测、预测和分类四个不同领域的数据挖掘问题。实验结果表明，针对不同数据特性选择合适的特征表示和模型架构是提升挖掘性能的关键。

## 参考文献
[1] TAN P N, STEINBACH M, KUMAR V, 等. 数据挖掘导论（原书第2版）[M]. 范明, 范宏建, 译. 北京: 机械工业出版社, 2020.  